# Evaluation Pipeline — Reference-Based Comparison

Compare synthesised audio (DDSP and baseline) against a **solo violin reference set**.

1. **Build reference** — extract and concatenate per-frame timbre features from all violin recordings.
2. **Strategy 1 (Distribution)** — compare concatenated folder-level distributions to the reference.
3. **Strategy 2 (Pairwise)** — compare each individual file to the reference distribution.
4. **Visualizations** — bar charts, box plots, heatmaps, method comparison.

In [1]:
import logging
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")

print(f"Project root: {PROJECT_ROOT}")

Project root: /m/home/home3/37/thieun1/unix/Project/final_project


In [2]:
from evaluation.timbre_metrics import TimbreMetrics
from evaluation.loss import Loss
from evaluation.experiment_pipeline import collect_wav_files
from visualize import (
    plot_distribution_comparison,
    plot_loss_by_group,
    plot_loss_boxplot,
    plot_loss_heatmap,
    plot_method_comparison,
)

import numpy as np
import pandas as pd

/u/37/thieun1/unix/anaconda3/envs/conda_env3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-05 17:48:30.087854: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-05 17:48:30.174700: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-05 17:48:38.369991: W tensorflow/compiler/xla/stream_executor/pla

## Paths

In [3]:
# --- Input / output paths ---
REFERENCE_CSV  = PROJECT_ROOT / "data" / "processed" / "bach_violin_timbre_features.csv"
DDSP_DIR       = PROJECT_ROOT / "data" / "processed" / "voice" / "Full_transfered"
BASELINE_DIR   = PROJECT_ROOT / "data" / "processed" / "voice" / "Full_baseline"

# --- Result CSVs ---
RESULTS_DIR    = PROJECT_ROOT / "artifacts" / "evaluation"

print(f"Reference CSV: {REFERENCE_CSV}  (exists: {REFERENCE_CSV.exists()})")
print(f"DDSP dir:      {DDSP_DIR}  (exists: {DDSP_DIR.exists()})")
print(f"Baseline dir:  {BASELINE_DIR}  (exists: {BASELINE_DIR.exists()})")
print(f"Results dir:   {RESULTS_DIR}")

Reference CSV: /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/bach_violin_timbre_features.csv  (exists: True)
DDSP dir:      /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_transfered  (exists: True)
Baseline dir:  /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_baseline  (exists: True)
Results dir:   /m/home/home3/37/thieun1/unix/Project/final_project/artifacts/evaluation


## 1. Load Pre-Extracted Reference Features

Load the reference timbre distribution from the pre-extracted CSV
(`bach_violin_timbre_features.csv`), which contains per-frame spectral
features for all Bach violin recordings.

In [4]:
SR = 16000
tm = TimbreMetrics(sample_rate=SR)
loss = Loss()

# --- Reference from pre-extracted CSV ---
ref_df = pd.read_csv(REFERENCE_CSV)
META_COLS = {"filename", "collection", "violinist", "work"}
feat_keys = sorted([c for c in ref_df.columns if c not in META_COLS])
ref_2d = ref_df[feat_keys].to_numpy(dtype=np.float64)

print(f"Reference: {ref_2d.shape[0]} frames, {ref_2d.shape[1]} features")
print(f"Feature keys: {feat_keys}")

Reference: 56132 frames, 11 features
Feature keys: ['mean_center', 'spectral_centroid', 'spectral_crest', 'spectral_decrease', 'spectral_energy', 'spectral_flatness', 'spectral_kurtosis', 'spectral_roll_off', 'spectral_skewness', 'spectral_slope', 'spectral_spread']


## 2. Strategy 1 — Distribution-Level Comparison

Compare the concatenated folder-level feature distribution of each method
against the reference.  Sub-sample to 10 000 frames for computational
efficiency (MMD is O(n^2)).

In [5]:
SYNTHESIZED_DIRS = {"ddsp": DDSP_DIR, "baseline": BASELINE_DIR}
MAX_FRAMES = 10_000
rng = np.random.default_rng(42)

dist_rows = []
synth_features_cache = {}  # reused by pairwise step

for method_name, synth_dir in SYNTHESIZED_DIRS.items():
    synth_features = tm.extract_from_dir(synth_dir)
    synth_features_cache[method_name] = synth_features
    synth_2d = np.column_stack([synth_features[k] for k in feat_keys])

    # Sub-sample for efficiency
    ref_sub = ref_2d
    synth_sub = synth_2d
    if ref_sub.shape[0] > MAX_FRAMES:
        ref_sub = ref_sub[rng.choice(ref_sub.shape[0], MAX_FRAMES, replace=False)]
    if synth_sub.shape[0] > MAX_FRAMES:
        synth_sub = synth_sub[rng.choice(synth_sub.shape[0], MAX_FRAMES, replace=False)]

    dist_result = loss.evaluate(synth_sub, ref_sub)
    dist_rows.append({
        "method": method_name,
        "n_files": len(collect_wav_files(synth_dir)),
        "total_frames": synth_2d.shape[0],
        **dist_result,
    })
    print(f"{method_name}: mmd={dist_result['mmd']:.6f}, wasserstein={dist_result['wasserstein']:.6f}")

df_distribution = pd.DataFrame(dist_rows)
display(df_distribution)

plot_distribution_comparison(df_distribution, "mmd")
plot_distribution_comparison(df_distribution, "wasserstein")

2026-04-05 17:49:23,576 INFO evaluation.timbre_metrics: Extracting features from 3095 files in /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_transfered
Feature extraction: 100%|██████████| 3095/3095 [03:36<00:00, 14.33it/s]
2026-04-05 17:52:59,643 INFO evaluation.timbre_metrics: Concatenated 3095 files -> 106805 frames, 11 features


ddsp: mmd=0.533927, wasserstein=626.651828


2026-04-05 17:53:14,978 INFO evaluation.timbre_metrics: Extracting features from 3095 files in /m/home/home3/37/thieun1/unix/Project/final_project/data/processed/voice/Full_baseline
Feature extraction: 100%|██████████| 3095/3095 [03:33<00:00, 14.47it/s]
2026-04-05 17:56:48,887 INFO evaluation.timbre_metrics: Concatenated 3095 files -> 106856 frames, 11 features


baseline: mmd=0.381646, wasserstein=546.706858


,method,n_files,total_frames,mmd,wasserstein
0,ddsp,3095,106805,0.533927,626.651828
1,baseline,3095,106856,0.381646,546.706858


## 3. Strategy 2 — Pairwise File-vs-Reference

Compare each individual synthesised file's feature distribution against the
full reference distribution.

In [ ]:
from tqdm.auto import tqdm

pair_rows = []

for method_name, synth_dir in SYNTHESIZED_DIRS.items():
    synth_dir_path = Path(synth_dir)
    wav_files = collect_wav_files(synth_dir_path)

    for wf in tqdm(wav_files, desc=f"Pairwise [{method_name}]"):
        rel = str(wf.relative_to(synth_dir_path))
        try:
            series = tm.extract_series_from_file(wf)
            if not series:
                continue

            available = sorted(set(series.keys()) & set(feat_keys))
            if not available:
                continue

            col_idx = [feat_keys.index(k) for k in available]
            file_2d = np.column_stack([series[k] for k in available])
            ref_subset = ref_2d[:, col_idx]

            distances = loss.evaluate(file_2d, ref_subset)
            pair_rows.append({"method": method_name, "file": rel, **distances})
        except Exception:
            continue

df_pairwise = pd.DataFrame(pair_rows)
print(f"Pairwise results: {len(df_pairwise)} rows")
df_pairwise.head()

Pairwise [ddsp]:   0%|          | 0/3095 [00:00<?, ?it/s]

### Pairwise Summary by Method

In [ ]:
display(df_pairwise.groupby("method")[["mmd", "wasserstein"]].describe().round(4))

plot_loss_boxplot(df_pairwise, "method", "mmd")
plot_loss_boxplot(df_pairwise, "method", "wasserstein")

## 4. Method Comparison

In [ ]:
plot_loss_by_group(df_pairwise, "method", "mmd")

In [ ]:
plot_loss_by_group(df_pairwise, "method", "wasserstein")

## 5. Export

Save summary tables and figures.

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# CSVs
df_distribution.to_csv(RESULTS_DIR / "distribution_summary.csv", index=False)
df_pairwise.to_csv(RESULTS_DIR / "evaluation_pairwise.csv", index=False)

pairwise_summary = df_pairwise.groupby("method")[["mmd", "wasserstein"]].agg(["mean", "std", "median"]).round(4)
pairwise_summary.to_csv(RESULTS_DIR / "pairwise_summary_by_method.csv")

# Figures
plot_distribution_comparison(df_distribution, "mmd", save_path=str(FIGURES_DIR / "distribution_mmd.png"))
plot_distribution_comparison(df_distribution, "wasserstein", save_path=str(FIGURES_DIR / "distribution_wasserstein.png"))
plot_loss_boxplot(df_pairwise, "method", "mmd", save_path=str(FIGURES_DIR / "pairwise_mmd_boxplot.png"))
plot_loss_boxplot(df_pairwise, "method", "wasserstein", save_path=str(FIGURES_DIR / "pairwise_wasserstein_boxplot.png"))
plot_loss_by_group(df_pairwise, "method", "mmd", save_path=str(FIGURES_DIR / "pairwise_mmd_by_method.png"))
plot_loss_by_group(df_pairwise, "method", "wasserstein", save_path=str(FIGURES_DIR / "pairwise_wasserstein_by_method.png"))

print(f"Exported to {RESULTS_DIR}")